# A full vetting run

The driver runs the tests you ask for over a table of candidates, merges every
JSON into one record per candidate, and writes a summary table with every
status, reason and vetting metric.

It runs on the two worked examples' light curves in `../examples`. With no
stellar parameter table (`PETCARE_STELLAR_CATALOG`), the physical tests take
their stellar density from the TIC. Tier 0 additionally queries Gaia, the TIC and ExoFOP over the
network, and caches every response under `PETCARE_CACHE`.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

OUT = Path("out_run")
OUT.mkdir(exist_ok=True)
CANDIDATES = OUT / "candidates.csv"

# The two worked examples' planets. Swap in your own search output: the driver
# reads any CSV with tic, period, t0 and duration columns. ra/dec are optional
# for the per-candidate tests (they are looked up), but pc_dedup needs them: it
# groups candidates by sky separation as well as by period and epoch.
import pandas as pd
EXAMPLES = Path("../examples")
table = pd.concat([pd.read_csv(EXAMPLES / ex / "candidates.csv")
                   for ex in ("ds_tuc", "hip67522")])
table.to_csv(CANDIDATES, index=False)
print(CANDIDATES.read_text())

## Run it

`--tiers 0 1` is the cheap pass: catalogue queries and light-curve tests only.
Tier 2 downloads pixels and tier 3 talks to ground-based archives, so both are
much slower and are usually run on a shortlist.

In [ ]:
cmd = [sys.executable, "-m", "petcare.scripts.petcare",
       "--candidates", str(CANDIDATES), "--tiers", "0", "1",
       "--lc-root", str(EXAMPLES / "ds_tuc" / "data"), str(EXAMPLES / "hip67522" / "data"),
       "--out", str(OUT)]
p = subprocess.run(cmd, capture_output=True, text=True)
print(p.stdout[-4000:] or p.stderr[-4000:])

## The summary table

One row per candidate, one column per test. This is the object you scan when
there are thousands of candidates; the per-candidate records are what you open
once a row looks interesting.

In [ ]:
import pandas as pd

summary = pd.read_csv(OUT / "summary.csv")
print(summary.shape)
summary[[c for c in summary.columns if not c.endswith("_reason")]].head(20)

In [ ]:
# which tests objected, and to what
status_cols = [c for c in summary.columns if c.endswith("_status")]
for _, row in summary.iterrows():
    print(f"\n{row.get('name', row['tic'])}")
    for c in status_cols:
        if row[c] in ("FLAG", "FAIL", "INCONCLUSIVE", "ERROR"):
            reason = str(row.get(c.replace("_status", "_reason"), ""))[:110]
            print(f"  {c[:-7]:12s} {row[c]:13s} {reason}")

**There is no overall score, deliberately.** Which test objected is the
information; a scalar hides it. `INCONCLUSIVE` in particular is neither a pass
nor a fail — it means the test could not be applied, usually because there were
too few covered epochs, and it should not be read as suspicion.

## The merged record

Everything each test measured, for one candidate, in one file.

In [ ]:
rec = json.loads(sorted(OUT.glob("*/petcare.json"))[0].read_text())
print(rec["target"]["name"], "|", rec["candidate"]["period"], "d\n")
for test, res in rec["tests"].items():
    print(f"{test:12s} {res['status']:13s} {res['reason'][:90]}")

## The plots and the metrics

Every test writes a plot beside its JSON, `<test>.png` for a star's first
candidate and `<test>_c1.png` for the second, and the plots are what to look at
before deciding anything. The numbers behind them are in each candidate's
`petcare.json` under `vetting_metrics`, and in `summary.csv` as `m_*` columns:
reported, never graded, for sorting and cutting a large run.

In [ ]:
from IPython.display import Image, display

metric_cols = [c for c in summary.columns if c.startswith("m_")]
print(len(metric_cols), "metrics, e.g.", metric_cols[:8])
display(summary[["name"] + metric_cols[:8]])

display(Image(filename=str(OUT / "TIC410214986" / "fit.png")))

## The cross-target tests

Two questions are not about a single candidate and are answered over the whole
list at once:

- **`pc_dedup`** — is the same signal appearing on several nearby stars? One
  bright eclipsing binary contaminates every star within a few arcminutes and
  produces an apparently independent candidate on each. Grouping by period,
  epoch *and* sky separation catches them wholesale, before any pixel work, for
  almost no cost. A single missed cluster puts a dozen fake candidates on a
  follow-up list.
- **`pc_known`** — is this already a catalogued TOI, TCE, eclipsing binary or
  variable star? It also flags the case where the candidate *is* a known
  planet, which is a useful check on the search itself.

Both annotate the candidate table rather than writing per-candidate records.

In [ ]:
for tool in ("pc_dedup", "pc_known"):
    p = subprocess.run([sys.executable, "-m", f"petcare.scripts.{tool}",
                        "--candidates", str(CANDIDATES), "--out", str(OUT)],
                       capture_output=True, text=True)
    print(p.stdout.strip()[-1500:] or p.stderr.strip()[-1500:])
    print("-" * 70)

## Where thresholds come from

Every threshold in the source carries a comment saying what it costs, measured
on a sample of known young planets and known false positives — or is marked
`PROVISIONAL` if it has not been measured yet. `calibration/samples/known_young_planets.csv`
holds the sample and `calibration/calibrate.py` measures each test's catch
rate and its cost in real planets, and scans thresholds.

That matters more here than in most vetting problems. Young stars break the
conventional numbers: odd/even at the textbook 3σ deletes 18% of the known
planets in this sample, and the two worst offenders — at 8.1σ and 7.2σ — are
both real planets whose epoch-to-epoch depths move because of spot crossings.
A threshold that has not been costed is not finished.